In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

TARGET_COL = 'prediction'  
ID_COL = 'patient_id'      

train_df = pd.read_csv('train.csv') 
test_df = pd.read_csv('test_features.csv')

train_df = train_df.fillna(train_df.median(numeric_only=True))
test_df = test_df.fillna(test_df.median(numeric_only=True))
train_df = train_df.replace([np.inf, -np.inf], 0)
test_df = test_df.replace([np.inf, -np.inf], 0)

test_ids = test_df[ID_COL].values

X = train_df.drop(columns=[TARGET_COL, ID_COL]).values.astype(np.float32)
y = train_df[TARGET_COL].values.astype(np.float32).reshape(-1, 1)
X_test = test_df.drop(columns=[ID_COL]).values.astype(np.float32)

print(f"Признаков: {X.shape[1]}, объектов: {X.shape[0]}")

scaler = StandardScaler() 
X_train = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)


X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

batch_size = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t), batch_size=batch_size, shuffle=False)


class DiabetesFFN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.bn1 = nn.BatchNorm1d(64) 
        self.fc2 = nn.Linear(64, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.fc3 = nn.Linear(32, 1) 
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BreastCancerFFN(X_train_t.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=0.0001)


epochs = 100
train_losses = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        
        has_nan = False
        for param in model.parameters():
            if param.grad is not None and (torch.isnan(param.grad).any() or torch.isinf(param.grad).any()):
                has_nan = True
                break
        if has_nan:
            optimizer.zero_grad()
            continue 
        
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
    
    epoch_train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {epoch_train_loss:.4f}")


plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss', color='blue')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Динамика потерь при обучении')
plt.legend()
plt.grid(True)
plt.show()


model.eval()
y_test_pred_prob = []
with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch[0].to(device)
        outputs = model(X_batch)
        probs = torch.sigmoid(outputs)
        probs = torch.nan_to_num(probs, nan=0.5)
        y_test_pred_prob.extend(probs.cpu().numpy().flatten())

y_test_pred_class = (np.array(y_test_pred_prob) >= 0.5).astype(int)


os.makedirs('outputs', exist_ok=True)
submission = pd.DataFrame({
    'patient_id': test_ids,
    'prediction': y_test_pred_class
})
submission.to_csv('outputs/task1_predictions.csv', index=False)
print("\nПервые 5 строк файла с предсказаниями:")
print(submission.head())